# Módulo 0.5 — Do sequenciador à matriz de contagens

**Workshop de RNA-seq · mieloma múltiplo**
Este módulo roda **antes do Dia 1**.

---

O Dia 1 começa com uma matriz de contagens pronta. Mas ela não nasce assim. Alguém
pegou os arquivos que saíram do sequenciador e rodou uma sequência de etapas até
chegar nela.

Aqui você vai rodar essas etapas. De verdade — não é simulação, não é vídeo.
Os comandos são os mesmos que um serviço de bioinformática usaria, e você vai ver
cada arquivo intermediário.

Leva **15 a 20 minutos**.

## Por que levedura, e não mieloma?

Porque os FASTQ do MMRF-CoMMpass são de **acesso controlado**, via dbGaP. A camada
aberta do GDC entrega contagens, não leituras.

Isso já é a primeira lição: **o nível de acesso ao dado determina em que ponto do
fluxo você consegue entrar.** Guarde isso — volta no Dia 2, quando o sexo dos
pacientes também não estiver disponível.

Vamos usar 4 amostras de *Saccharomyces cerevisiae* do conjunto de teste do nf-core
(GSE110004), subamostradas para poucos MB. O genoma da levedura tem 12 Mb contra
3 Gb do humano — é o que permite construir o índice em um minuto em vez de uma hora.

## O caminho

```
FASTQ  →  QC  →  trimagem  →  QC de novo  →  alinhamento  →  BAM  →  contagem  →  MATRIZ
 (1)      (2)      (3)          (3.5)          (4)          (5)      (6)          ↓
                                                                          aqui começa o Dia 1
                    └──────────────────────────────────────────────┘
                          MultiQC junta os relatórios de todas essas etapas
```

Repare no passo **3.5**: rodar o QC de novo depois da trimagem não é redundância.
São perguntas diferentes. Antes: *como está o dado?* Depois: *a trimagem fez o que
eu esperava, e só o que eu esperava?*

> ⚠️ **Ferramentas de linha de comando, não de R ou Python.** FastQC, fastp,
> HISAT2, samtools e featureCounts são programas independentes. Este notebook só
> os chama. É por isso que esta etapa roda no Colab, que é Linux, e não na sua
> máquina Windows.

---
## 1 · Instalar as ferramentas

Cinco programas, direto do repositório do Ubuntu. Leva 1 a 2 minutos.

Cada um faz uma coisa só, e é assim que a bioinformática funciona: você encadeia
programas pequenos em vez de usar um único que faz tudo.

| Programa | Papel no fluxo |
|---|---|
| `fastqc` | diagnóstico das leituras — mede, não corrige |
| `fastp` | remove adaptador e apara pontas ruins |
| `hisat2` | alinha as leituras no genoma, atravessando íntrons |
| `samtools` | manipula os arquivos de alinhamento (SAM/BAM) |
| `subread` | traz o `featureCounts`, que conta leituras por gene |

In [ ]:
%%bash
# apt-get update  → atualiza a lista de pacotes disponíveis
# -y              → responde "sim" a tudo (não há ninguém para digitar)
# -qq             → modo quieto; a saída do apt não interessa aqui
# O redirecionamento > /dev/null 2>&1 joga fora saída e erros do apt.
apt-get update -qq > /dev/null 2>&1
apt-get install -y -qq fastqc fastp hisat2 samtools subread > /dev/null 2>&1
echo "instalação concluída"

In [ ]:
# Conferir se cada ferramenta responde. Se alguma falhar, o resto não roda —
# melhor descobrir agora do que no meio do alinhamento.
import shutil
import subprocess
import sys

FERRAMENTAS = {
    "fastqc":       ["fastqc", "--version"],
    "fastp":        ["fastp", "--version"],
    "hisat2":       ["hisat2", "--version"],
    "hisat2-build": ["hisat2-build", "--version"],
    "samtools":     ["samtools", "--version"],
    "featureCounts":["featureCounts", "-v"],
}

print("Verificando ferramentas de bioinformática...\n")

faltando = []
erros = []

for nome, cmd in FERRAMENTAS.items():
    if shutil.which(cmd[0]) is None:
        faltando.append(nome)
        print(f"  ❌ {nome:<14} não encontrado")
        continue

    try:
        r = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            errors='replace',
            timeout=10
        )

        versao = (r.stdout + r.stderr).strip().split("\n")[0][:60]
        print(f"  ✅ {nome:<14} {versao}")

    except subprocess.TimeoutExpired:
        erros.append(f"{nome} (timeout)")
        print(f"  ⚠️  {nome:<14} timeout ao executar")
    except Exception as e:
        erros.append(f"{nome} ({str(e)})")
        print(f"  ⚠️  {nome:<14} erro: {type(e).__name__}")

print("\n" + "="*60)

if faltando or erros:
    msg = []
    if faltando:
        msg.append(f"❌ Não instaladas: {', '.join(faltando)}")
    if erros:
        msg.append(f"⚠️  Erros ao verificar: {', '.join(erros)}")

    print("\n".join(msg))
    print("\n❌ Não é seguro prosseguir. Instale as ferramentas faltantes.")
    sys.exit(1)
else:
    print("✅ Todas as ferramentas foram encontradas e respondem.")
    print("✅ Seguro prosseguir com a análise.")
    print("\n" + "="*60)

---
## 2 · Baixar os dados

Quatro amostras, o genoma e a anotação. Tudo do repositório público de dados de
teste do nf-core.

| Amostra | Grupo | O que é |
|---|---|---|
| SRR6357070 | WT | levedura selvagem, réplica 1 |
| SRR6357072 | WT | levedura selvagem, réplica 2 |
| SRR6357073 | RAP1 | linhagem RAP1, sem indução |
| SRR6357076 | RAP1 | linhagem RAP1, 30 min de auxina |

Usamos só a leitura R1 de cada uma (*single-end*). O fluxo com leitura pareada é
o mesmo, com um arquivo a mais por amostra.

In [ ]:
%%bash
set -e            # aborta na primeira falha; sem isso, um download quebrado
                  # passa despercebido e o erro só aparece no alinhamento

mkdir -p dados/fastq dados/ref

BASE=https://raw.githubusercontent.com/nf-core/test-datasets/rnaseq

# O teste [ ! -s arquivo ] significa "não existe OU está vazio".
# Com ele, reexecutar a célula não rebaixa nada — importante num auditório
# com 30 pessoas na mesma rede.
for SRR in SRR6357070 SRR6357072 SRR6357073 SRR6357076; do
  if [ ! -s dados/fastq/${SRR}.fastq.gz ]; then
    wget -q -O dados/fastq/${SRR}.fastq.gz \
      ${BASE}/testdata/GSE110004/${SRR}_1.fastq.gz
  fi
done

# O genoma (FASTA) diz qual é a sequência de cada cromossomo.
# A anotação (GTF) diz onde começam e terminam genes e éxons.
# São dois arquivos diferentes e são necessários os dois: o alinhador usa o
# primeiro, o contador usa o segundo.
[ -s dados/ref/genoma.fa ] || \
  wget -q -O dados/ref/genoma.fa https://github.com/nf-core/test-datasets/raw/rnaseq/reference/genome.fasta
[ -s dados/ref/genes.gtf ] || {
  wget -q -O dados/ref/genes.gtf.gz https://github.com/nf-core/test-datasets/raw/rnaseq/reference/genes.gtf.gz
  gunzip -f dados/ref/genes.gtf.gz
}

echo "=== arquivos baixados ==="
ls -lh dados/fastq/ dados/ref/

In [ ]:
# Trava: arquivo truncado aqui vira erro incompreensível no alinhamento.
from pathlib import Path
import gzip

AMOSTRAS = {"SRR6357070": "WT", "SRR6357072": "WT",
            "SRR6357073": "RAP1", "SRR6357076": "RAP1"}

problemas = []
for srr in AMOSTRAS:
    p = Path(f"dados/fastq/{srr}.fastq.gz")
    if not p.exists() or p.stat().st_size < 10_000:
        problemas.append(f"{srr}: ausente ou pequeno demais"); continue
    try:                                     # o gzip abre até o fim?
        with gzip.open(p, "rt") as fh:
            n = sum(1 for _ in fh)
        if n % 4:
            problemas.append(f"{srr}: {n} linhas, não é múltiplo de 4 — truncado")
        else:
            print(f"  ✅ {srr}  {p.stat().st_size/1e6:.1f} MB  {n//4:,} leituras".replace(",", "."))
    except Exception as e:
        problemas.append(f"{srr}: {type(e).__name__} ao descompactar")

for nome, minimo in [("dados/ref/genoma.fa", 1e5), ("dados/ref/genes.gtf", 1e4)]:
    p = Path(nome)
    if not p.exists() or p.stat().st_size < minimo:
        problemas.append(f"{nome}: ausente ou incompleto")
    else:
        print(f"  ✅ {nome}  {p.stat().st_size/1e6:.2f} MB")

if problemas:
    raise RuntimeError("Download incompleto:\n  " + "\n  ".join(problemas) +
                       "\n\nRode a célula anterior de novo.")
print("\nTodos os arquivos íntegros.")

---
## 3 · Anatomia de um FASTQ

Este é o arquivo que sai do sequenciador. Antes de processar, **olhe**.

In [ ]:
!zcat dados/fastq/SRR6357070.fastq.gz | head -8

### O que você está vendo

São **quatro linhas por leitura**, sempre nessa ordem:

| Linha | Conteúdo |
|---|---|
| 1 | identificador, começa com `@` |
| 2 | a sequência de bases |
| 3 | separador, começa com `+` |
| 4 | a qualidade de cada base, **um caractere por base** |

A linha 4 é a que quase ninguém entende. Cada caractere é um número codificado em
ASCII — o **Phred score**. A conta é:

$$Q = -10 \times \log_{10}(P_{\text{erro}})$$

Um `Q30` significa 1 chance em 1.000 de a base estar errada. Um `Q20`, 1 em 100.

A célula abaixo decodifica a qualidade da primeira leitura, caractere por caractere.

In [ ]:
import gzip

with gzip.open("dados/fastq/SRR6357070.fastq.gz", "rt") as fh:
    ident, seq, _, qual = [next(fh).rstrip() for _ in range(4)]

print(f"identificador : {ident}")
print(f"comprimento   : {len(seq)} bases\n")
print("  base  caractere  Phred   prob. de erro")
print("  " + "-" * 44)
for i in range(min(12, len(seq))):
    q = ord(qual[i]) - 33          # Illumina usa offset 33
    p = 10 ** (-q / 10)
    print(f"   {seq[i]}        {qual[i]}        {q:>3}     1 em {1/p:>10,.0f}".replace(",", "."))

phred = [ord(ch) - 33 for ch in qual]
print(f"\n  Phred médio desta leitura : {sum(phred)/len(phred):.1f}")
print(f"  menor Phred               : {min(phred)}")
print(f"  bases com Q >= 30         : {100*sum(q >= 30 for q in phred)/len(phred):.0f}%")

print("""
💬 Repare que a qualidade costuma cair no FIM da leitura. Isso é característico
   do sequenciamento por síntese: a cada ciclo, uma fração das moléculas do
   cluster fica fora de fase, e o sinal vai perdendo nitidez.""")

---
## 4 · Controle de qualidade — FastQC

O FastQC lê o FASTQ inteiro e produz um relatório com uma dúzia de gráficos. Não
corrige nada: só mostra.

In [ ]:
%%bash
mkdir -p qc/fastqc

# -q       modo quieto (sem barra de progresso)
# -t 2     processa 2 arquivos em paralelo (o Colab dá 2 núcleos)
# -o       pasta de saída
#
# Para cada FASTQ saem dois arquivos: um .html (o relatório que se abre no
# navegador) e um .zip (os mesmos dados em texto e os gráficos em PNG — é do
# zip que as próximas células leem, e é ele que o MultiQC vai ler na seção 9).
fastqc -q -t 2 -o qc/fastqc dados/fastq/*.fastq.gz

ls qc/fastqc/

In [ ]:
# O relatório é um HTML, mas dentro do .zip vem também o resumo em texto e os
# gráficos em PNG. Vamos ler os dois.
import zipfile, glob
from pathlib import Path
from IPython.display import display, Image, Markdown

resumos = {}
for z in sorted(glob.glob("qc/fastqc/*_fastqc.zip")):
    nome = Path(z).stem.replace("_fastqc", "")
    with zipfile.ZipFile(z) as zf:
        raiz = zf.namelist()[0].split("/")[0]
        txt = zf.read(f"{raiz}/summary.txt").decode()
        resumos[nome] = [l.split("\t")[:2] for l in txt.strip().split("\n")]

modulos = [m for _, m in resumos[list(resumos)[0]]]
print(f"{'módulo':<38}" + "".join(f"{n[-4:]:>8}" for n in resumos))
print("-" * (38 + 8 * len(resumos)))
simbolo = {"PASS": "  ✅", "WARN": "  ⚠️", "FAIL": "  ❌"}
for i, m in enumerate(modulos):
    linha = f"{m[:37]:<38}"
    for nome in resumos:
        linha += f"{simbolo.get(resumos[nome][i][0], '  ?'):>8}"
    print(linha)

print("""
💬 ⚠️ e ❌ no FastQC NÃO significam dado ruim. Os limites são genéricos, pensados
   para sequenciamento de genoma. Em RNA-seq, três módulos falham quase sempre —
   e por motivos biológicos, não técnicos:

   · Per base sequence content  — o início da leitura tem viés dos primers
                                   aleatórios usados na síntese do cDNA
   · Sequence duplication levels — gene muito expresso gera leituras idênticas.
                                   Isso é sinal, não artefato
   · Overrepresented sequences   — mesma coisa: rRNA e transcritos abundantes

   Olhe o relatório, não o semáforo.""")

In [ ]:
# O gráfico que importa: qualidade por posição na leitura.
with zipfile.ZipFile("qc/fastqc/SRR6357070_fastqc.zip") as zf:
    raiz = zf.namelist()[0].split("/")[0]
    for img in ["per_base_quality.png", "per_sequence_quality.png"]:
        try:
            with open(img, "wb") as fh:
                fh.write(zf.read(f"{raiz}/Images/{img}"))
            display(Markdown(f"**{img}**"))
            display(Image(img))
        except KeyError:
            print(f"({img} não está neste relatório)")

---
## 5 · Trimagem — fastp

Remove adaptadores e apara as pontas de baixa qualidade.

> ⚠️ **Trimagem agressiva é prática superada.** Durante anos se cortou tudo abaixo
> de Q30. Hoje se sabe que isso **piora** a quantificação: você joga fora leituras
> boas e enviesa a cobertura. Os alinhadores modernos fazem *soft-clipping* — eles
> ignoram as pontas ruins sozinhos.
>
> Williams *et al.* (2016), *BMC Bioinformatics*, mostrou isso com números.
>
> Aqui usamos parâmetros conservadores de propósito. Repare no fim quantas
> leituras foram descartadas: pouquíssimas.

In [ ]:
%%bash
set -e
mkdir -p limpo qc/fastp

for SRR in SRR6357070 SRR6357072 SRR6357073 SRR6357076; do
  fastp \
    -i dados/fastq/${SRR}.fastq.gz \
    -o limpo/${SRR}.fastq.gz \
    --qualified_quality_phred 15 \
    --length_required 25 \
    --json qc/fastp/${SRR}.json \
    --html qc/fastp/${SRR}.html \
    --thread 2 2> /dev/null
done
echo "trimagem concluída"

# -i / -o                        entrada e saída. Como o dado é single-end, há
#                                um de cada. Em paired-end seriam -i/-I e -o/-O.
# --qualified_quality_phred 15   uma base conta como "boa" a partir de Q15.
#                                Q15 = 1 erro a cada ~32 bases. Parece frouxo, e
#                                é de propósito: veja a caixa acima.
# --length_required 25           leitura que sobrar com menos de 25 bases é
#                                descartada. Abaixo disso ela alinha em qualquer
#                                lugar por acaso e só gera ruído.
# --json / --html                dois relatórios do mesmo conteúdo. O JSON é para
#                                a máquina ler (próxima célula, e o MultiQC);
#                                o HTML é para você.
#
# O que NÃO está aqui e é o ponto principal: nenhuma flag de adaptador. O fastp
# detecta a sequência do adaptador sozinho, olhando a sobreposição das leituras.
# Você raramente precisa dizer qual é — e, quando digita, costuma errar.

In [ ]:
# O fastp grava no JSON o estado ANTES e DEPOIS do filtro, no mesmo arquivo.
# É aí que se mede o efeito real da trimagem — não no gráfico, no número.
import json as _json

print(f"{'amostra':<14}{'leituras antes':>16}{'depois':>12}{'descartado':>13}{'Q30 antes':>12}{'Q30 depois':>12}")
print("-" * 79)
for srr in AMOSTRAS:
    with open(f"qc/fastp/{srr}.json") as fh:
        j = _json.load(fh)
    # summary.before_filtering e summary.after_filtering trazem total_reads,
    # total_bases, q20_rate, q30_rate, read1_mean_length, gc_content...
    antes, depois = j["summary"]["before_filtering"], j["summary"]["after_filtering"]
    perda = 100 * (1 - depois["total_reads"] / antes["total_reads"])
    print(f"{srr:<14}{antes['total_reads']:>16,}{depois['total_reads']:>12,}"
          f"{perda:>12.1f}%{100*antes['q30_rate']:>11.1f}%{100*depois['q30_rate']:>11.1f}%"
          .replace(",", "."))

print("""
💬 A perda é pequena, e o Q30 sobe pouco. É exatamente esse o ponto: com dados
   modernos, a trimagem quase não muda nada. Ela existe para o caso em que há
   adaptador de verdade — e é o fastp quem detecta isso, não você.

   Guarde estes números. Na próxima seção você vai ver os mesmos arquivos pelos
   olhos do FastQC, e a conclusão tem que ser a mesma. Quando duas ferramentas
   independentes discordam sobre o mesmo arquivo, uma das duas está sendo mal
   usada.""")

---
### 5.1 · O mesmo FastQC, agora no arquivo limpo

Rodar o FastQC antes da trimagem responde *como está o dado*. Rodar depois
responde outra pergunta, e é a que quase todo mundo pula: **a trimagem fez o que
você esperava — e só o que você esperava?**

É a mesma ferramenta, no mesmo dado, depois do fastp. A comparação lado a lado é
o produto desta seção.

**O que procurar:**

| Onde | Antes | Depois, se a trimagem funcionou |
|---|---|---|
| Qualidade por posição | cai no fim da leitura | a queda no fim é aparada |
| Distribuição de comprimento | um pico só, tudo do mesmo tamanho | espalha um pouco à esquerda |
| Conteúdo de adaptador | sobe no fim, se houver adaptador | encosta no zero |
| Semáforo ⚠️/❌ | três módulos falham | **falham os mesmos três** |

A última linha é a mais importante. Se o semáforo mudou muito, a trimagem foi
agressiva demais e você jogou dado fora. Se não mudou quase nada — que é o
esperado aqui —, está certo.

In [ ]:
%%bash
set -e

# ---------------------------------------------------------------------------
# FastQC nos arquivos JÁ TRIMADOS.
#
# Detalhe que parece bobo e não é: o FastQC batiza a amostra com o nome do
# arquivo. Como limpo/SRR6357070.fastq.gz tem o MESMO nome do arquivo cru, os
# dois relatórios sairiam com o mesmo rótulo — e o MultiQC, na seção 9, juntaria
# os dois como se fossem uma amostra só, sobrescrevendo um com o outro.
#
# Solução: criar links com o sufixo _trimado e rodar o FastQC em cima deles.
# Link, não cópia — não duplica um byte de dado.
# ---------------------------------------------------------------------------
mkdir -p qc/fastqc_limpo nomes_trimados

for SRR in SRR6357070 SRR6357072 SRR6357073 SRR6357076; do
  ln -sf ../limpo/${SRR}.fastq.gz nomes_trimados/${SRR}_trimado.fastq.gz
done

# -q = silencioso   -t 2 = duas amostras em paralelo   -o = pasta de saída
fastqc -q -t 2 -o qc/fastqc_limpo nomes_trimados/*.fastq.gz

echo "=== relatórios do depois ==="
ls qc/fastqc_limpo/

In [ ]:
# ---------------------------------------------------------------------------
# Comparação numérica antes × depois.
#
# Dentro de cada .zip do FastQC há um fastqc_data.txt: o MESMO conteúdo do HTML,
# em texto tabulado. É de lá que sai todo gráfico do relatório — e é de lá que o
# MultiQC lê. Aprender a ler esse arquivo é o que permite automatizar o QC.
#
# Estrutura do arquivo:
#     >>Nome do módulo    PASS|WARN|FAIL
#     #cabeçalho das colunas
#     linhas de dados, separadas por TAB
#     >>END_MODULE
# ---------------------------------------------------------------------------
import zipfile
from pathlib import Path


def ler_modulo(zip_path, nome_modulo):
    "Devolve as linhas de dados de um módulo do fastqc_data.txt, já em colunas."
    with zipfile.ZipFile(zip_path) as zf:
        raiz = zf.namelist()[0].split("/")[0]
        txt = zf.read(f"{raiz}/fastqc_data.txt").decode("utf-8", "replace")
    dentro, linhas = False, []
    for l in txt.split("\n"):
        if l.startswith(">>" + nome_modulo):
            dentro = True
            continue
        if dentro and l.startswith(">>END_MODULE"):
            break
        if dentro and l and not l.startswith("#"):
            linhas.append(l.split("\t"))
    return linhas


def ler_flags(zip_path):
    "PASS/WARN/FAIL de cada módulo, na ordem em que o FastQC os escreve."
    with zipfile.ZipFile(zip_path) as zf:
        raiz = zf.namelist()[0].split("/")[0]
        txt = zf.read(f"{raiz}/summary.txt").decode()
    return [l.split("\t")[:2] for l in txt.strip().split("\n")]


def ler_basico(zip_path, chave):
    for c in ler_modulo(zip_path, "Basic Statistics"):
        if c[0] == chave:
            return c[1]
    return ""


ZIP_ANTES  = {s: f"qc/fastqc/{s}_fastqc.zip"              for s in AMOSTRAS}
ZIP_DEPOIS = {s: f"qc/fastqc_limpo/{s}_trimado_fastqc.zip" for s in AMOSTRAS}

print("Qualidade média por posição — o número que o gráfico mostra, resumido\n")
print(f"{'amostra':<13}{'Q médio (todas as posições)':^30}{'Q na última posição':^26}")
print(f"{'':<13}{'antes  →  depois':^30}{'antes  →  depois':^26}")
print("-" * 69)

for srr in AMOSTRAS:
    # coluna 0 = posição na leitura, coluna 1 = qualidade média naquela posição
    qa = [float(c[1]) for c in ler_modulo(ZIP_ANTES[srr],  "Per base sequence quality")]
    qd = [float(c[1]) for c in ler_modulo(ZIP_DEPOIS[srr], "Per base sequence quality")]
    print(f"{srr:<13}"
          f"{f'{sum(qa)/len(qa):.1f}  →  {sum(qd)/len(qd):.1f}':^30}"
          f"{f'{qa[-1]:.1f}  →  {qd[-1]:.1f}':^26}")

print("\n\nComprimento das leituras — a assinatura da trimagem\n")
print(f"{'amostra':<13}{'nº de leituras':^28}{'comprimento':^24}")
print(f"{'':<13}{'antes  →  depois':^28}{'antes  →  depois':^24}")
print("-" * 65)
for srr in AMOSTRAS:
    na = ler_basico(ZIP_ANTES[srr],  "Total Sequences")
    nd = ler_basico(ZIP_DEPOIS[srr], "Total Sequences")
    ca = ler_basico(ZIP_ANTES[srr],  "Sequence length")
    cd = ler_basico(ZIP_DEPOIS[srr], "Sequence length")
    na = f"{int(na):,}".replace(",", ".")
    nd = f"{int(nd):,}".replace(",", ".")
    print(f"{srr:<13}{f'{na}  →  {nd}':^28}{f'{ca}  →  {cd}':^24}")

print("""
💬 Repare no campo "comprimento". Antes da trimagem costuma ser um número só —
   todas as leituras saíram do sequenciador com o mesmo tamanho. Depois vira um
   intervalo (ex.: 25-51), porque cada leitura perdeu um pedaço diferente.

   Esse é o rastro mais honesto de que a trimagem aconteceu.""")

# ---------------------------------------------------------------------------
# O semáforo, antes e depois, para as quatro amostras de uma vez.
# ---------------------------------------------------------------------------
flags_a = {s: ler_flags(ZIP_ANTES[s])  for s in AMOSTRAS}
flags_d = {s: ler_flags(ZIP_DEPOIS[s]) for s in AMOSTRAS}
simbolo = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌"}
modulos = [m for _, m in flags_a[list(AMOSTRAS)[0]]]

print("\n\nSemáforo do FastQC — 4 amostras, antes e depois\n")
print(f"{'módulo':<36}{'antes':<14}{'depois':<14}")
print("-" * 64)
for i, m in enumerate(modulos):
    a = "".join(simbolo.get(flags_a[s][i][0], "?") for s in AMOSTRAS)
    d = "".join(simbolo.get(flags_d[s][i][0], "?") for s in AMOSTRAS)
    mudou = "  ← mudou" if a != d else ""
    print(f"{m[:35]:<36}{a:<14}{d:<14}{mudou}")

print("""
💬 A leitura correta desta tabela é: quase nada mudou — e está certo.

   Os módulos que continuam em ⚠️/❌ falham por biologia (viés dos primers
   aleatórios, duplicação de transcritos abundantes), não por qualidade técnica.
   Nenhuma trimagem conserta isso, porque não há nada para consertar.

   Se você mexer no --qualified_quality_phred para 30 e rodar de novo, vai ver
   esta tabela mudar bastante — e a taxa de alinhamento, lá na frente, NÃO vai
   melhorar na mesma proporção. É o experimento sugerido no fim do notebook.""")

In [ ]:
# ---------------------------------------------------------------------------
# Os gráficos do FastQC, antes e depois, lado a lado.
#
# Os PNG já vêm prontos dentro do .zip, na pasta Images/. Não estamos redesenhando
# nada: é exatamente a figura que você veria abrindo o HTML no navegador.
#
# Para trocar o que é exibido, edite a lista GRAFICOS. Os nomes disponíveis são:
#   per_base_quality.png            qualidade por posição   ← o principal
#   per_sequence_quality.png        distribuição de Q por leitura
#   sequence_length_distribution.png comprimento das leituras
#   adapter_content.png             adaptador ao longo da leitura
#   per_base_sequence_content.png   viés de base por posição
#   duplication_levels.png          duplicação
# ---------------------------------------------------------------------------
import base64, zipfile
from IPython.display import HTML, display

GRAFICOS = [
    ("per_base_quality.png",             "Qualidade por posição na leitura"),
    ("sequence_length_distribution.png", "Distribuição de comprimento"),
    ("adapter_content.png",              "Conteúdo de adaptador"),
]

LARGURA = 400   # px de cada figura; aumente se a tela for grande


def png_b64(zip_path, img):
    "Extrai um PNG de dentro do zip e devolve em base64, para embutir no HTML."
    with zipfile.ZipFile(zip_path) as zf:
        raiz = zf.namelist()[0].split("/")[0]
        return base64.b64encode(zf.read(f"{raiz}/Images/{img}")).decode()


for img, titulo in GRAFICOS:
    html = [f"<h4 style='font-family:sans-serif;margin:18px 0 6px'>{titulo}"
            f" <span style='font-weight:400;color:#777'>({img})</span></h4>",
            "<table style='border-collapse:collapse'>",
            "<tr>"
            "<th style='padding:4px'></th>"
            "<th style='padding:4px;font-family:sans-serif'>ANTES — arquivo cru</th>"
            "<th style='padding:4px;font-family:sans-serif'>DEPOIS — pós-fastp</th>"
            "</tr>"]
    algum = False
    for srr in AMOSTRAS:
        celulas = []
        for z in (f"qc/fastqc/{srr}_fastqc.zip",
                  f"qc/fastqc_limpo/{srr}_trimado_fastqc.zip"):
            try:
                b64 = png_b64(z, img)
                celulas.append(f"<td style='padding:2px'><img src='data:image/png;base64,"
                               f"{b64}' width='{LARGURA}'></td>")
                algum = True
            except KeyError:
                celulas.append("<td style='padding:2px;color:#999;font-family:sans-serif'>"
                               "(este gráfico não existe neste relatório)</td>")
        html.append(f"<tr><td style='padding:4px;font-family:monospace'>{srr}"
                    f"<br><span style='color:#777'>{AMOSTRAS[srr]}</span></td>"
                    + "".join(celulas) + "</tr>")
    html.append("</table>")
    if algum:
        display(HTML("".join(html)))

print("""
💬 No gráfico de qualidade por posição, o fundo tem três faixas: verde (Q≥28),
   amarela (20-28) e vermelha (<20). A linha azul é a média; a caixa é o
   intervalo interquartil.

   Com estes dados a diferença entre antes e depois é sutil, e é para ser mesmo.
   O gráfico onde a trimagem aparece de verdade é o de comprimento: antes, uma
   barra única; depois, uma cauda à esquerda.

   O de adaptador quase não sai do zero — este conjunto de teste praticamente não
   tem adaptador residual. Num dado real com biblioteca de inserto curto, essa
   curva sobe até 20-30% no fim da leitura, e é a razão de existir a trimagem.""")

---
## 6 · Alinhamento — HISAT2

Cada leitura precisa ser localizada no genoma. O desafio do RNA-seq é que uma
leitura pode **atravessar uma junção de éxons** — metade dela cai num éxon e a
outra metade em outro, com um íntron de milhares de bases no meio.

Alinhador de DNA não sabe fazer isso. HISAT2 e STAR sabem: são *splice-aware*.

O primeiro passo é construir o índice do genoma. Para a levedura leva menos de um
minuto. Para o genoma humano levaria mais de uma hora e uns 200 GB de RAM com o
STAR — é por isso que ninguém refaz o índice: você baixa pronto.

In [ ]:
%%bash
set -e
mkdir -p indice

# hisat2-build transforma o FASTA num índice FM (6 arquivos .ht2). O alinhador
# não lê o FASTA: ele lê o índice. É uma estrutura que responde "onde está esta
# sequência de 20 bases no genoma?" em tempo praticamente constante.
#
# -p 2  usa 2 núcleos.
# O if evita reconstruir o índice se a célula for rodada duas vezes.
if [ ! -s indice/levedura.1.ht2 ]; then
  hisat2-build -p 2 dados/ref/genoma.fa indice/levedura > /dev/null 2>&1
fi
echo "=== índice construído ==="
ls -lh indice/ | head -5

echo ""
echo "=== o que tem no genoma de referência ==="
# Cada linha com > é uma sequência: aqui, cromossomos da levedura. Num genoma
# humano completo haveria também scaffolds não montados e o mitocondrial.
grep "^>" dados/ref/genoma.fa | head
echo ""
grep -c "^>" dados/ref/genoma.fa | xargs echo "sequências:"

In [ ]:
%%bash
set -e
mkdir -p bam qc/hisat2

for SRR in SRR6357070 SRR6357072 SRR6357073 SRR6357076; do
  hisat2 -p 2 -x indice/levedura -U limpo/${SRR}.fastq.gz \
    --summary-file qc/hisat2/${SRR}.txt \
    2> /dev/null \
  | samtools sort -@ 2 -o bam/${SRR}.bam -
  samtools index bam/${SRR}.bam
done

echo "=== BAM gerados ==="
ls -lh bam/*.bam
ls -lh bam/*.bai | head -1

# -x                índice (o prefixo, sem o .1.ht2)
# -U                leituras não pareadas (single-end). Em paired-end: -1 e -2.
# --summary-file    grava as estatísticas de alinhamento em arquivo. É esse
#                   arquivo que a próxima célula lê — e que o MultiQC lê na
#                   seção 9. Sem ele, o resumo iria para a tela e se perderia.
#
# O | é o detalhe que mais importa aqui. O hisat2 escreve SAM (texto) na saída
# padrão; em vez de gravar esse texto em disco, mandamos direto para o samtools
# sort, que ordena por posição no genoma e grava BAM (binário, comprimido).
# Para um genoma humano isso economiza dezenas de GB de escrita por amostra.
#
# samtools index cria o .bai, um índice que permite pular direto para uma região
# do genoma sem varrer o arquivo. É o que o IGV usa.

### A taxa de alinhamento é o primeiro número que se olha

Abaixo de 70% em RNA-seq há algo errado: contaminação, genoma de referência
trocado, ou adaptador que não foi removido.

In [ ]:
# O arquivo do --summary-file é texto livre. Em vez de contar linhas na mão,
# extraímos os dois números que interessam com expressão regular — que é como o
# MultiQC faz para as 100+ ferramentas que ele entende.
import re

print(f"{'amostra':<14}{'leituras':>12}{'alinhadas':>12}{'taxa':>9}   situação")
print("-" * 62)
for srr in AMOSTRAS:
    txt = open(f"qc/hisat2/{srr}.txt").read()
    total = int(re.search(r"(\d+) reads; of these", txt).group(1))
    taxa  = float(re.search(r"([\d.]+)% overall alignment rate", txt).group(1))
    alinhadas = round(total * taxa / 100)
    marca = "✅ boa" if taxa >= 70 else ("⚠️ baixa — investigue" if taxa >= 50
                                        else "❌ algo está errado")
    print(f"{srr:<14}{total:>12,}{alinhadas:>12,}{taxa:>8.1f}%   {marca}".replace(",", "."))

print("""
💬 O que derruba a taxa de alinhamento, em ordem de frequência:
   1. genoma de referência errado (organismo ou versão)
   2. adaptador não removido — a leitura não casa com nada
   3. contaminação por outro organismo
   4. rRNA em excesso, se a biblioteca não foi depletada

   Note que "leituras" aqui é o número DEPOIS da trimagem: o hisat2 recebeu os
   arquivos de limpo/, não os de dados/fastq/. Comparar este total com o da
   tabela do fastp fecha a contabilidade — nenhuma leitura some sem explicação.""")

---
## 7 · O que é um BAM

BAM é a versão binária e comprimida do SAM. Uma linha por alinhamento, com onde a
leitura caiu, quão bem casou, e a leitura em si.

Você nunca abre um BAM direto — usa o `samtools` para traduzir.

In [ ]:
%%bash
echo "=== três alinhamentos, em formato legível ==="
samtools view bam/SRR6357070.bam | head -3
echo ""
echo "=== resumo do arquivo ==="
samtools flagstat bam/SRR6357070.bam

### Lendo uma linha do SAM

As primeiras colunas são o essencial:

| Coluna | O que é |
|---|---|
| 1 | nome da leitura |
| 2 | *flag* — bits que dizem se alinhou, se é reversa, se é duplicada |
| 3 | em que cromossomo caiu |
| 4 | em que posição |
| 5 | MAPQ — confiança do alinhamento |
| 6 | CIGAR — o mapa do alinhamento |

O **CIGAR** é o campo mais informativo. Um `50M` significa 50 bases casadas em
sequência. Um `30M2000N20M` significa 30 bases num éxon, 2.000 bases puladas
(o íntron), e 20 bases no éxon seguinte.

**O `N` é a assinatura de uma junção de éxons.** É o que só um alinhador
*splice-aware* produz.

In [ ]:
# Quantas leituras atravessaram junções? Procuramos o N no CIGAR.
import subprocess, collections

r = subprocess.run(["samtools", "view", "bam/SRR6357070.bam"],
                   capture_output=True, text=True)
linhas = r.stdout.strip().split("\n")

cigars = [l.split("\t")[5] for l in linhas if len(l.split("\t")) > 5]
com_juncao = [c for c in cigars if "N" in c]

print(f"alinhamentos            : {len(cigars):,}".replace(",", "."))
print(f"atravessando junção (N) : {len(com_juncao):,}".replace(",", ".")
      + f"  ({100*len(com_juncao)/max(len(cigars),1):.1f}%)")

if com_juncao:
    print("\nexemplos de CIGAR com junção:")
    for c in com_juncao[:5]:
        print("   ", c)
else:
    print("""
   Nenhuma junção nesta amostra. É esperado: a levedura tem pouquíssimos íntrons
   — cerca de 300 genes em 6.000. No humano, quase todo gene tem, e a fração de
   leituras com N passa de 30%.""")

print("\ndistribuição de MAPQ (confiança do alinhamento):")
mapq = collections.Counter(int(l.split("\t")[4]) for l in linhas if len(l.split("\t")) > 4)
for q, n in sorted(mapq.items(), reverse=True)[:6]:
    print(f"   MAPQ {q:>3} : {n:>7,}".replace(",", ".")
          + f"   {'(único e confiável)' if q >= 30 else '(multimapeada ou ambígua)'}")

---
## 8 · Contagem — featureCounts

Agora a pergunta muda: não é mais "onde essa leitura caiu?", e sim **"quantas
leituras caíram dentro de cada gene?"**

Para responder, é preciso saber onde ficam os genes. É isso que o GTF diz.

In [ ]:
%%bash
echo "=== como é o GTF ==="
grep -v "^#" dados/ref/genes.gtf | head -3
echo ""
echo "=== quantos genes e éxons ==="
awk -F'\t' '$3=="gene"' dados/ref/genes.gtf | wc -l | xargs echo "genes:"
awk -F'\t' '$3=="exon"' dados/ref/genes.gtf | wc -l | xargs echo "éxons:"

In [ ]:
%%bash
set -e
mkdir -p contagens

featureCounts \
  -T 2 \
  -a dados/ref/genes.gtf \
  -t exon \
  -g gene_id \
  -o contagens/matriz.txt \
  bam/*.bam 2> contagens/log.txt

echo "=== resumo da atribuição ==="
cat contagens/matriz.txt.summary

# -T 2        núcleos
# -a          a anotação. É ela que define o que é um gene — troque o GTF e a
#             matriz muda, com os mesmos BAM. Por isso o Dia 1 registra a versão
#             do GENCODE.
# -t exon     conta leituras que caem em ÉXONS. É o padrão em RNA-seq: o mRNA
#             maduro não tem íntron, então leitura em íntron é pré-mRNA ou ruído.
# -g gene_id  soma os éxons pelo gene a que pertencem. Se fosse transcript_id,
#             a contagem seria por transcrito — e aí featureCounts é a ferramenta
#             errada (use salmon ou kallisto).
# -o          saída. Gera DOIS arquivos: matriz.txt (as contagens) e
#             matriz.txt.summary (o resumo abaixo).
#
# Sem -s: assumimos biblioteca não-orientada. Se a biblioteca for stranded e você
# esquecer o -s 2, metade das leituras vira Unassigned_NoFeatures. É o erro mais
# comum desta etapa, e o resumo abaixo é onde ele aparece.

### O que significa cada linha do resumo

| Categoria | O que aconteceu |
|---|---|
| `Assigned` | caiu dentro de um gene — é o que vira contagem |
| `Unassigned_NoFeatures` | alinhou no genoma, mas fora de qualquer gene |
| `Unassigned_Ambiguity` | caiu na sobreposição de dois genes |
| `Unassigned_MultiMapping` | alinhou em vários lugares |

**Você já viu essas categorias antes.** No Dia 1, ao abrir o arquivo STAR do GDC,
as quatro primeiras linhas eram `N_unmapped`, `N_multimapping`, `N_noFeature` e
`N_ambiguous` — e nós as descartamos. São exatamente estas, com outro nome.

Agora você sabe de onde vinham.

In [ ]:
# ---------------------------------------------------------------------------
# O mesmo resumo acima, mas: (1) com percentuais, (2) legendado e (3) salvo em
# arquivo de texto junto com os dados.
#
# Por que salvar? Porque o resumo é metadado do dado. Daqui a seis meses, quando
# alguém perguntar por que uma amostra tem metade das contagens das outras, a
# resposta está aqui — e não na tela de um notebook que já foi fechado.
# ---------------------------------------------------------------------------
from pathlib import Path
from datetime import datetime

ARQ_RESUMO = Path("resumo_atribuicao.txt")

LEGENDA = {
    "Assigned":                  "caiu dentro de um gene — é o que vira contagem",
    "Unassigned_Unmapped":       "não alinhou no genoma",
    "Unassigned_Read_Type":      "tipo de leitura incompatível com os parâmetros",
    "Unassigned_Singleton":      "par incompleto (só em paired-end)",
    "Unassigned_MappingQuality": "MAPQ abaixo do mínimo exigido",
    "Unassigned_Chimera":        "as duas pontas em cromossomos diferentes",
    "Unassigned_FragmentLength": "fragmento fora do intervalo esperado",
    "Unassigned_Duplicate":      "marcada como duplicata de PCR",
    "Unassigned_MultiMapping":   "alinhou igualmente bem em vários lugares",
    "Unassigned_Secondary":      "alinhamento secundário do mesmo par",
    "Unassigned_NonSplit":       "leitura não dividida, com --countSplitAlignmentsOnly",
    "Unassigned_NoFeatures":     "alinhou no genoma, mas fora de qualquer gene",
    "Unassigned_Overlapping_Length": "sobreposição com o gene curta demais",
    "Unassigned_Ambiguity":      "caiu na sobreposição de dois genes",
}

# --- ler o .summary -------------------------------------------------------
linhas = [l.split("\t") for l in
          Path("contagens/matriz.txt.summary").read_text().strip().split("\n")]
cabecalho, dados = linhas[0], linhas[1:]
amostras = [Path(c).stem for c in cabecalho[1:]]          # bam/SRR...bam → SRR...
tabela = {l[0]: [int(v) for v in l[1:]] for l in dados}
totais = [sum(tabela[k][i] for k in tabela) for i in range(len(amostras))]

# mantemos Assigned sempre; das demais, só as que têm ao menos uma leitura
categorias = ["Assigned"] + [k for k in tabela
                             if k != "Assigned" and sum(tabela[k]) > 0]

# --- montar o texto -------------------------------------------------------
L = max(len(c) for c in categorias) + 2
COL = 22

out = []
out.append("RESUMO DA ATRIBUIÇÃO DE LEITURAS — featureCounts")
out.append("Módulo 0.5 · Workshop de RNA-seq · dados de teste nf-core (GSE110004)")
out.append("")
out.append(f"gerado em      : {datetime.now():%Y-%m-%d %H:%M}")
out.append("anotação       : dados/ref/genes.gtf")
out.append("feature contada: exon      agrupada por: gene_id")
out.append("alinhador      : HISAT2 (leituras single-end, pós-fastp)")
out.append("")
out.append("=" * (L + COL * len(amostras)))
out.append(f"{'categoria':<{L}}" + "".join(f"{a:>{COL}}" for a in amostras))
out.append("=" * (L + COL * len(amostras)))

for cat in categorias:
    linha = f"{cat:<{L}}"
    for i in range(len(amostras)):
        n = tabela[cat][i]
        pct = 100 * n / totais[i] if totais[i] else 0
        linha += f"{f'{n:,}'.replace(',', '.') + f'  ({pct:4.1f}%)':>{COL}}"
    out.append(linha)

out.append("-" * (L + COL * len(amostras)))
out.append(f"{'TOTAL':<{L}}" + "".join(
    f"{f'{t:,}'.replace(',', '.'):>{COL}}" for t in totais))
out.append("")
out.append("LEGENDA")
for cat in categorias:
    out.append(f"  {cat:<{L}}{LEGENDA.get(cat, '')}")
out.append("")
out.append("COMO LER")
out.append("  · Assigned abaixo de ~60% em RNA-seq pede investigação. As causas")
out.append("    mais comuns são: strandness errada (-s), anotação de outra versão")
out.append("    do genoma, ou biblioteca com muito rRNA/DNA genômico.")
out.append("  · NoFeatures alto com alinhamento alto = as leituras estão no genoma,")
out.append("    mas fora dos genes anotados. Suspeite da anotação, não do dado.")
out.append("  · MultiMapping alto é típico de famílias gênicas e pseudogenes; no")
out.append("    humano fica em 10-20% e é normal.")
out.append("  · Estas mesmas categorias aparecem no arquivo do STAR que o Dia 1")
out.append("    baixa do GDC, com outros nomes: N_unmapped, N_multimapping,")
out.append("    N_noFeature, N_ambiguous.")

texto = "\n".join(out)
ARQ_RESUMO.write_text(texto, encoding="utf-8")

print(texto)
print(f"\n💾 salvo em: {ARQ_RESUMO.resolve()}")

---
## 9 · MultiQC — um relatório para todos

Conte quantos arquivos de QC você gerou até aqui, para **quatro** amostras:

| Origem | Arquivos |
|---|---|
| FastQC antes | 4 HTML + 4 ZIP |
| FastQC depois | 4 HTML + 4 ZIP |
| fastp | 4 HTML + 4 JSON |
| HISAT2 | 4 sumários |
| featureCounts | 1 resumo |

São 29 arquivos. Com 60 amostras — o tamanho do nosso subconjunto do CoMMpass —
seriam mais de 400. Ninguém abre 400 relatórios.

O **MultiQC** varre um diretório, reconhece o formato de saída de mais de 100
ferramentas de bioinformática e junta tudo em **um HTML só**: uma linha por
amostra na tabela geral, e um gráfico por métrica com todas as amostras
sobrepostas.

**É o primeiro arquivo que se abre quando o serviço de sequenciamento entrega os
dados.** A amostra que destoa aparece na hora — uma curva fora do feixe, uma
linha vermelha na tabela.

> Ele não é instalado pelo `apt` como os outros: é um pacote Python. Por isso a
> instalação leva um minuto a mais.

In [ ]:
# O MultiQC é escrito em Python, então vem pelo pip e não pelo apt.
# -q reduz a saída; ainda assim leva ~1 minuto, porque ele traz plotly junto.
!pip install -q multiqc
!multiqc --version

In [ ]:
%%bash
# multiqc <diretórios a varrer> --outdir <onde gravar>
#
# Passamos qc/ e contagens/. Ele desce recursivamente, abre cada arquivo, tenta
# reconhecer o formato e ativa o módulo correspondente. Nada é configurado à mão:
# o FastQC é reconhecido pelo fastqc_data.txt dentro do zip, o fastp pelo JSON,
# o HISAT2 pela frase "overall alignment rate", o featureCounts pelo .summary.
#
# --force  sobrescreve um relatório anterior (permite reexecutar a célula)
# O diretório de saída é relatorio/, FORA de qc/, para o MultiQC não tentar
# varrer o próprio relatório numa segunda execução.
multiqc qc contagens \
  --outdir relatorio \
  --filename relatorio_multiqc \
  --force

echo ""
echo "=== o que foi gerado ==="
ls -lh relatorio/
ls relatorio/relatorio_multiqc_data/ 2>/dev/null | head -20

In [ ]:
# ---------------------------------------------------------------------------
# O MultiQC não gera só o HTML: junto vai uma pasta *_data com TUDO em texto.
# É a parte que quase ninguém usa, e é a mais útil — dá para versionar no git,
# comparar entre corridas, alimentar um script.
#
# multiqc_general_stats.txt é a tabela do topo do relatório, uma linha por
# amostra. Aqui ela entra num DataFrame.
# ---------------------------------------------------------------------------
import glob, re
import pandas as pd
from IPython.display import display

arqs = glob.glob("relatorio/**/multiqc_general_stats.txt", recursive=True)

if not arqs:
    print("Tabela geral não encontrada — veja o HTML diretamente na pasta relatorio/.")
else:
    geral = pd.read_csv(arqs[0], sep="\t").set_index("Sample")
    # os nomes de coluna vêm longos, do tipo
    # "FastQC_mqc-generalstats-fastqc-percent_duplicates" — encurtamos
    geral.columns = [re.sub(r".*mqc-generalstats-[^-]+-", "", c) for c in geral.columns]
    geral = geral.dropna(axis=1, how="all").round(2)

    pd.set_option("display.width", 200, "display.max_columns", 50)
    print(f"Tabela geral do MultiQC — {geral.shape[0]} linhas × {geral.shape[1]} métricas\n")
    display(geral)

    print("""
💬 Repare que cada amostra aparece MAIS DE UMA VEZ, com métricas diferentes:
   uma linha vinda do FastQC do arquivo cru, outra do FastQC pós-trimagem
   (sufixo _trimado), outra do fastp, outra do HISAT2.

   É por isso que renomeamos os arquivos trimados lá na seção 5.1. Sem aquele
   sufixo, o MultiQC teria colapsado o antes e o depois na mesma linha, e a
   comparação — que é o objetivo de rodar o FastQC duas vezes — sumiria.

   Esse é um erro real e frequente em pipeline de produção.""")

In [ ]:
# ---------------------------------------------------------------------------
# Abrir o relatório. O HTML é autocontido (traz o JavaScript dentro), então
# funciona offline, num pendrive, anexado num e-mail.
#
# No Colab a forma confiável de vê-lo é baixar. Se o download for bloqueado pelo
# navegador, use o ícone de pasta 📁 na barra esquerda → relatorio/ →
# clique com o botão direito em relatorio_multiqc.html → Download.
# ---------------------------------------------------------------------------
from pathlib import Path

REL = Path("relatorio/relatorio_multiqc.html")

if not REL.exists():
    raise SystemExit("Relatório não encontrado — rode a célula do multiqc acima.")

print(f"{REL}  —  {REL.stat().st_size/1e6:.1f} MB")

try:
    from google.colab import files
    files.download(str(REL))
    print("\nDownload iniciado. Abra o arquivo no seu navegador.")
except Exception as e:
    print(f"\n(download automático indisponível: {type(e).__name__})")
    print("Use o painel de arquivos 📁 à esquerda para baixar relatorio/relatorio_multiqc.html")

print("""
🔎 O que olhar quando abrir:

   1. General Statistics — a tabela do topo. Ordene por "% Aligned" e veja se
      alguma amostra destoa. Em dado real é aqui que a amostra problemática
      aparece em 5 segundos.

   2. FastQC: Mean Quality Scores — todas as curvas sobrepostas. As linhas do
      antes e do _trimado juntas mostram o efeito da trimagem em uma figura só.

   3. FastQC: Sequence Length Distribution — antes, um espeto; depois, uma cauda.

   4. HISAT2: Alignment Scores — barras empilhadas por amostra.

   5. featureCounts: Assignments — a mesma informação do resumo_atribuicao.txt,
      em barra proporcional.

   Passe o mouse sobre qualquer ponto: ele diz de que amostra é.""")

---
## 10 · A matriz

É aqui que o Dia 1 começa.

In [ ]:
# ---------------------------------------------------------------------------
# O arquivo do featureCounts não é uma matriz limpa: as seis primeiras colunas
# são anotação (Geneid, Chr, Start, End, Strand, Length) e só da sétima em diante
# vêm as amostras. Além disso, o nome da coluna é o caminho do BAM inteiro.
#
# Arrumar isso é a última etapa antes do Dia 1.
# ---------------------------------------------------------------------------
import pandas as pd
from pathlib import Path

bruta = pd.read_csv("contagens/matriz.txt", sep="\t", comment="#")
print("colunas que o featureCounts devolve:")
print("  ", list(bruta.columns[:6]), "...\n")

# Geneid vira índice; iloc[:, 5:] descarta as 5 colunas de anotação restantes
mat = bruta.set_index("Geneid").iloc[:, 5:]
mat.columns = [Path(c).stem for c in mat.columns]   # bam/SRR6357070.bam → SRR6357070
mat = mat[list(AMOSTRAS)]                           # ordem estável, não a do glob

# A tabela de metadados é tão importante quanto a matriz: é ela que diz qual
# amostra é de qual grupo. Sem isso, a matriz é um monte de números.
meta = pd.DataFrame({"grupo": [AMOSTRAS[s] for s in mat.columns]}, index=mat.columns)

print(f"Matriz: {mat.shape[0]:,} genes × {mat.shape[1]} amostras".replace(",", "."))
print(f"\nprofundidade por amostra (leituras atribuídas):")
for s in mat.columns:
    print(f"   {s}  {AMOSTRAS[s]:<5} {mat[s].sum():>10,}".replace(",", "."))

# Duas conferências que valem por dez: gene sem nenhuma contagem não carrega
# informação, e o filtro do Dia 1 (>=10 contagens) já pode ser antecipado aqui.
print(f"\ngenes com contagem zero em todas : {(mat.sum(axis=1) == 0).sum():,}".replace(",", "."))
print(f"genes com pelo menos 10 no total : {(mat.sum(axis=1) >= 10).sum():,}".replace(",", "."))

mat.to_csv("matriz_contagens.csv")
meta.to_csv("metadata.csv")

print("""
=== arquivos finais deste módulo ===
   matriz_contagens.csv    genes × amostras — a entrada do Dia 1
   metadata.csv            qual amostra pertence a qual grupo
   resumo_atribuicao.txt   de onde vieram (e para onde foram) as leituras
   relatorio/relatorio_multiqc.html   o QC de todas as etapas, num arquivo

Os quatro juntos são o mínimo para alguém reproduzir esta análise. Uma matriz
sem os outros três é um número sem procedência.""")

mat.head(10)

In [ ]:
# Última conferência antes do Dia 1: as amostras do mesmo grupo se parecem?
#
# log2(x+1) porque contagem tem distribuição muito assimétrica — um punhado de
# genes com dezenas de milhares de leituras domina qualquer correlação em escala
# linear. O +1 evita log de zero.
# Spearman (ordem) em vez de Pearson (valor) pelo mesmo motivo.
import numpy as np, matplotlib.pyplot as plt

log = np.log2(mat[mat.sum(axis=1) >= 10] + 1)
corr = log.corr(method="spearman")

fig, ax = plt.subplots(figsize=(5.5, 4.6))
im = ax.imshow(corr, cmap="RdYlBu_r", vmin=corr.values.min(), vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_xticklabels([f"{c[-4:]}\n{AMOSTRAS[c]}" for c in corr.columns], fontsize=9)
ax.set_yticks(range(len(corr)))
ax.set_yticklabels([f"{c[-4:]} {AMOSTRAS[c]}" for c in corr.index], fontsize=9)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.3f}", ha="center", va="center",
                fontsize=8.5, color="black")
ax.set_title("Correlação de Spearman entre amostras", fontsize=11, fontweight="bold")
fig.colorbar(im, fraction=.046)
plt.tight_layout(); plt.show()

print("""
💬 Amostras do mesmo grupo deveriam se parecer mais entre si do que com as do
   outro grupo. Se não se parecem, o problema está no dado ou nos rótulos — e
   nenhuma estatística conserta isso depois.

   Com 4 amostras e um subconjunto do genoma, não espere separação perfeita.

   No Dia 1 esta mesma pergunta volta com outro nome — PCA — e com o cuidado
   extra de usar VST em vez de log2(x+1). O raciocínio é idêntico; muda a
   transformação.""")

---
## 11 · O que você acabou de fazer

```
FASTQ            leituras cruas do sequenciador, 4 linhas por leitura
  ↓ FastQC       diagnóstico do dado bruto — não corrige nada
  ↓ fastp        adaptadores e pontas ruins, com parcimônia
  ↓ FastQC       de novo, no arquivo limpo — a trimagem fez o que devia?
  ↓ HISAT2       localiza cada leitura no genoma, atravessando íntrons
  ↓ samtools     ordena e indexa
  ↓ featureCounts conta quantas leituras caíram em cada gene
MATRIZ           genes × amostras
  +
MultiQC          junta os relatórios de todas as etapas acima num arquivo só
```

**Foi exatamente isso que o consórcio MMRF fez com 800 amostras**, com STAR no
lugar do HISAT2 e o genoma humano no lugar do da levedura. O resultado é o arquivo
que o Dia 1 baixa do GDC.

### Quatro coisas para levar

**1. A matriz de contagens não é o dado bruto.** Ela é o produto de cinco decisões:
qual genoma, qual anotação, qual alinhador, quais parâmetros de trimagem, qual
critério de contagem. Todas afetam o resultado, e nenhuma aparece na matriz.

**2. Por isso o release importa.** Quando o Dia 1 registra "GDC Data Release 46.0",
não é burocracia — é a única forma de outra pessoa refazer estas etapas do
mesmo jeito.

**3. QC não é uma etapa, é um hábito.** Você mediu o dado antes da trimagem,
depois da trimagem, depois do alinhamento e depois da contagem. Cada medida
respondeu a uma pergunta diferente, e as quatro juntas — no MultiQC — são o que
permite dizer "este dado está bom" com alguma honestidade.

**4. Nível de acesso define onde você entra no fluxo.** Não pudemos usar os FASTQ
do mieloma porque são controlados. Isso não é detalhe administrativo: determina
que análises são possíveis.

---

### 🔧 Experimente

1. Volte à célula do fastp e mude `--qualified_quality_phred` de 15 para 30.
   Rode dali para baixo, incluindo o FastQC pós-trimagem e o MultiQC. Quantas
   leituras a mais foram descartadas? O semáforo da seção 5.1 mudou? E a taxa de
   alinhamento melhorou o suficiente para compensar a perda?

2. Na célula do featureCounts, troque `-t exon` por `-t gene`. Compare os dois
   `resumo_atribuicao.txt`: o que muda em `Assigned` e em `Unassigned_NoFeatures`,
   e por quê?

3. Ainda no featureCounts, acrescente `-s 1` (biblioteca orientada no sentido da
   fita). Veja `Unassigned_NoFeatures` explodir. É esse o rastro do parâmetro de
   strandness errado — o erro mais caro e mais silencioso desta etapa.

4. Rode `samtools view bam/SRR6357070.bam | awk '$5 < 10' | wc -l` para contar as
   leituras de baixa confiança. Elas entraram na contagem?

---

*Material didático. Dados de teste do nf-core (GSE110004), Saccharomyces
cerevisiae. As ferramentas usadas — FastQC, fastp, HISAT2, samtools, Subread e
MultiQC — são todas de código aberto.*